In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50) 

In [4]:
df = pd.read_csv('../data/raw/city_day.csv') 
print(df.shape) 
display(df.head()) 
print(df.dtypes) 

(29531, 16)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


City              str
Date              str
PM2.5         float64
PM10          float64
NO            float64
NO2           float64
NOx           float64
NH3           float64
CO            float64
SO2           float64
O3            float64
Benzene       float64
Toluene       float64
Xylene        float64
AQI           float64
AQI_Bucket        str
dtype: object


In [ ]:
df['Date'] = pd.to_datetime(df['Date']) 
hyd = ( 
    df[df['City'].str.strip().str.lower() == 'hyderabad'] 
    .sort_values('Date') 
    .drop_duplicates(subset=['Date']) 
    .set_index('Date') 
)

In [6]:
print(hyd.index.min(), hyd.index.max()) 
print(hyd.shape) 
display(hyd.head()) 

2015-01-04 00:00:00 2020-07-01 00:00:00
(2006, 15)


,City,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
Date,,,,,,,,,,,,,,,
2015-01-04,Hyderabad,NaN,NaN,3.70,17.19,20.89,NaN,0.30,2.58,30.34,0.41,6.15,1.11,NaN,NaN
2015-01-05,Hyderabad,NaN,NaN,3.61,9.69,13.30,NaN,0.23,5.19,31.28,0.39,7.22,0.55,NaN,NaN
2015-01-06,Hyderabad,NaN,NaN,4.21,22.02,26.23,NaN,0.31,9.09,28.56,0.52,5.14,0.53,NaN,NaN
2015-01-07,Hyderabad,NaN,NaN,26.08,40.08,66.15,NaN,0.53,9.03,31.08,1.92,16.35,2.14,NaN,NaN
2015-01-08,Hyderabad,NaN,NaN,10.31,33.02,43.33,NaN,0.44,8.47,35.72,1.89,14.27,1.58,NaN,NaN


In [7]:
missing = hyd.isna().mean().sort_values(ascending=False) 
display((missing * 100).round(1).to_frame('missing_percent')) 

,missing_percent
NH3,17.9
PM10,17.7
Benzene,6.3
Toluene,6.3
Xylene,6.3
AQI,6.3
AQI_Bucket,6.3
PM2.5,5.7
NO2,1.4
NO,1.4


In [8]:
wanted = ['AQI', 'PM2.5', 'PM10', 'NO2', 'CO', 'O3'] 
hyd = hyd[wanted].copy() 
 
for column in wanted: 
    hyd[column] = pd.to_numeric(hyd[column], errors='coerce') 
 
print(hyd.describe().T) 
 

        count        mean        std    min      25%     50%       75%     max
AQI    1880.0  109.207447  53.142709  22.00  75.0000  104.00  130.0000  737.00
PM2.5  1891.0   47.035357  38.522589   4.83  25.7900   42.00   61.4150  571.02
PM10   1651.0   91.931532  40.476131  10.54  61.5250   94.48  117.4450  485.88
NO2    1978.0   28.386754  15.924444   0.62  15.4775   25.57   38.8675   92.33
CO     2001.0    0.590780   0.515106   0.00   0.3200    0.57    0.7700    8.83
O3     1980.0   33.611005  14.936788   0.02  22.5150   32.11   43.1125  107.68


In [9]:
full_dates = pd.date_range(hyd.index.min(), hyd.index.max(), freq='D') 
hyd = hyd.reindex(full_dates) 
hyd.index.name = 'Date' 

missing_before = hyd.isna().sum() 
hyd = hyd.interpolate(method='time', limit=2) 
hyd = hyd.dropna(subset=['AQI']) 
missing_after = hyd.isna().sum() 
 
print(pd.DataFrame({ 
    'before': missing_before, 
    'after': missing_after 
})) 

       before  after
AQI       126      0
PM2.5     115      0
PM10      355    235
NO2        28      0
CO          5      0
O3         26      0


In [11]:
assert hyd.index.is_monotonic_increasing 
assert not hyd.index.duplicated().any() 
assert hyd['AQI'].notna().all()

negative_counts = (hyd.select_dtypes('number') < 0).sum() 
print(negative_counts) 
print(hyd.shape)

AQI      0
PM2.5    0
PM10     0
NO2      0
CO       0
O3       0
dtype: int64
(1897, 6)


In [14]:
hyd.to_csv('../data/processed/hyderabad_aqi.csv') 
print('Saved:', '../data/processed/hyderabad_aqi.csv')

Saved: ../data/processed/hyderabad_aqi.csv
